# Week 4, Day 3 - more LangGraph..

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [ ]:
# Our favorite first step! Crew was doing this for us, by the way.
load_dotenv(override=True)

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper

serper = GoogleSerperAPIWrapper()
serper.run("What is the capital of France?")

## Now here is a LangChain wrapper class for converting functions into Tools

In [ ]:
from langchain.tools import BaseTool
from typing import Callable

class CustomTool(BaseTool):
    name: str
    description: str
    func: Callable[[str], str]
    
    def _run(self, query: str):
        return self.func(query)

tool_search = CustomTool(name="search", description = "Useful for online search", func=serper.run)

In [ ]:
tool_search.invoke("What is the capital of France?")

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

In [ ]:
tool_push = CustomTool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

tool_push.invoke("Hello, me")